# Extract ratings for matched movies

Parse Netflix `combined_data_*.txt` files (README format) and keep only ratings for movies present in `movies_metadata.csv`. Drop users with fewer than 5 ratings among those matched movies. Save the result to `data/processed/ratings.csv`.

## 1. Load allow-list

Read `netflix_movie_id` from the processed metadata join.

In [1]:
from pathlib import Path
import csv

import pandas as pd

ROOT = Path("..")
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

allowed = set(
    pd.read_csv(PROCESSED / "movies_metadata.csv", usecols=["netflix_movie_id"])[
        "netflix_movie_id"
    ].astype(int)
)
print(f"Allowed movies: {len(allowed):,}")

Allowed movies: 1,806


## 2. Stream-parse combined_data files

Each file uses movie headers `MovieID:` followed by `RaterID,Rating,Date` lines. Stream line-by-line and write only ratings whose movie id is in the allow-list.

In [2]:
from collections import Counter

combined_files = sorted(RAW.glob("combined_data_*.txt"))
out_path = PROCESSED / "ratings.csv"

n_rows = 0
n_movies_kept = 0
movies_seen = set()
user_counts = Counter()
rating_min = None
rating_max = None

with open(out_path, "w", newline="", encoding="utf-8") as out_f:
    writer = csv.writer(out_f)
    writer.writerow(["netflix_movie_id", "rater_id", "rating", "date"])

    for path in combined_files:
        print(f"Parsing {path.name}...")
        keep = False
        movie_id = None
        with open(path, encoding="utf-8") as in_f:
            for line in in_f:
                line = line.rstrip("\n")
                if line.endswith(":"):
                    movie_id = int(line[:-1])
                    keep = movie_id in allowed
                    if keep and movie_id not in movies_seen:
                        movies_seen.add(movie_id)
                        n_movies_kept += 1
                    continue
                if not keep:
                    continue
                rater_id, rating, date = line.split(",")
                rater_id_int = int(rater_id)
                rating_int = int(rating)
                writer.writerow([movie_id, rater_id_int, rating_int, date])
                user_counts[rater_id_int] += 1
                n_rows += 1
                if rating_min is None or rating_int < rating_min:
                    rating_min = rating_int
                if rating_max is None or rating_int > rating_max:
                    rating_max = rating_int

print(f"Wrote {n_rows:,} ratings for {n_movies_kept:,} movies to {out_path}")
print(f"Unique users: {len(user_counts):,}")
print(f"Rating range: {rating_min}–{rating_max}")
print(f"Movies in allow-list with no ratings: {len(allowed - movies_seen):,}")

Parsing combined_data_1.txt...
Parsing combined_data_2.txt...
Parsing combined_data_3.txt...
Parsing combined_data_4.txt...
Wrote 57,222,299 ratings for 1,806 movies to ..\data\processed\ratings.csv
Unique users: 477,927
Rating range: 1–5
Movies in allow-list with no ratings: 0


## 3. User rating-count buckets

How many users fall into each ratings-count range (before the sparse-user filter). Use this to choose `MIN_USER_RATINGS`.

In [8]:
import numpy as np

ratings_per_user = np.fromiter(user_counts.values(), dtype=np.int32)

bins = [0, 5, 10, 25, 50, 100, 250, 500, np.inf]
labels = ["1–5", "6–10", "11–25", "26–50", "51–100", "101–250", "251–500", "501+"]
bucket = pd.cut(ratings_per_user, bins=bins, labels=labels, right=True)

bucket_panel = (
    bucket.value_counts()
    .reindex(labels, fill_value=0)
    .rename_axis("ratings")
    .reset_index(name="n_users")
)
bucket_panel["pct_users"] = (
    100 * bucket_panel["n_users"] / bucket_panel["n_users"].sum()
).round(1)

display(bucket_panel)
print(
    f"Users: {len(ratings_per_user):,} | "
    f"mean: {ratings_per_user.mean():.1f} | "
    f"median: {np.median(ratings_per_user):.0f} | "
    f"max: {ratings_per_user.max():,}"
)

,ratings,n_users,pct_users
0,1–5,17926,3.8
1,6–10,22540,4.7
2,11–25,80303,16.8
3,26–50,93732,19.6
4,51–100,86871,18.2
5,101–250,107669,22.5
6,251–500,53766,11.2
7,501+,15120,3.2


Users: 477,927 | mean: 119.7 | median: 61 | max: 1,789


## 4. Filter sparse users

Keep only users who rated at least `MIN_USER_RATINGS` matched movies. Stream-rewrite `ratings.csv` so the full file never loads into memory.

In [9]:
MIN_USER_RATINGS = 5

eligible_users = {uid for uid, n in user_counts.items() if n >= MIN_USER_RATINGS}
n_users_before = len(user_counts)
n_users_after = len(eligible_users)
n_rows_before = n_rows

tmp_path = out_path.with_suffix(".csv.tmp")
n_rows_after = 0
movies_after = set()

with open(out_path, encoding="utf-8") as in_f, open(
    tmp_path, "w", newline="", encoding="utf-8"
) as out_f:
    reader = csv.reader(in_f)
    writer = csv.writer(out_f)
    header = next(reader)
    writer.writerow(header)
    for row in reader:
        if int(row[1]) in eligible_users:
            writer.writerow(row)
            movies_after.add(int(row[0]))
            n_rows_after += 1

tmp_path.replace(out_path)

print(
    f"Users: {n_users_before:,} → {n_users_after:,} "
    f"(dropped {n_users_before - n_users_after:,} with <{MIN_USER_RATINGS} ratings)"
)
print(f"Ratings: {n_rows_before:,} → {n_rows_after:,}")
print(f"Movies: {len(movies_seen):,} → {len(movies_after):,}")
print(f"Wrote filtered ratings to {out_path}")

Users: 477,927 → 463,657 (dropped 14,270 with <5 ratings)
Ratings: 57,222,299 → 57,186,566
Movies: 1,806 → 1,806
Wrote filtered ratings to ..\data\processed\ratings.csv


## 5. Sanity check

Peek at the filtered output without loading the full file into memory.

In [10]:
sample = pd.read_csv(out_path, nrows=5)
display(sample)
print(f"File size: {out_path.stat().st_size / (1024 ** 2):.1f} MB")
print(f"Users kept: {n_users_after:,} / {n_users_before:,}")
print(f"Ratings kept: {n_rows_after:,} / {n_rows_before:,}")
print(f"Rated movies kept: {len(movies_after)}")

,netflix_movie_id,rater_id,rating,date
0,3,1025579,4,2003-03-29
1,3,712664,5,2004-02-01
2,3,1331154,4,2004-07-03
3,3,2632461,3,2005-07-22
4,3,44937,5,2004-06-22


File size: 1472.2 MB
Users kept: 463,657 / 477,927
Ratings kept: 57,186,566 / 57,222,299
Rated movies kept: 1806
